### базовая 23.0934%

In [ ]:
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

import lightgbm as lgb

In [ ]:
train_df = pd.read_csv("/content/train (1).csv")
test_df = pd.read_csv("/content/test_x (1).csv")

target_col = "salary_mean_net"

id_col = (
    'id'
    if 'id' in test_df.columns
    else ('ID' if 'ID' in test_df.columns else test_df.columns[0])
)

In [ ]:
text_cols = [
    "name_clean",
    "lemmaized_wo_stopwords_raw_description",
    "key_skills_name"
]

cat_cols = [
    "experience_name",
    "schedule_name",
    "employment_name",
    "unified_address_city",
    "unified_address_state",
    "professional_roles_name"
]

bool_cols = [
    "accept_handicapped",
    "accept_kids"
]

for col in text_cols + cat_cols:
    train_df[col] = train_df[col].fillna("не указано").astype(str)
    test_df[col] = test_df[col].fillna("не указано").astype(str)

In [ ]:
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

train_cat = encoder.fit_transform(train_df[cat_cols])
test_cat = encoder.transform(test_df[cat_cols])

train_bool = train_df[bool_cols].astype(int).values
test_bool = test_df[bool_cols].astype(int).values

In [ ]:
vec_name = TfidfVectorizer(max_features=5000,ngram_range=(1, 2))
train_name_tfidf = vec_name.fit_transform(train_df["name_clean"])
test_name_tfidf = vec_name.transform(test_df["name_clean"])
vec_desc = TfidfVectorizer(max_features=10000,min_df=5)

train_desc_tfidf = vec_desc.fit_transform(train_df["lemmaized_wo_stopwords_raw_description"])
test_desc_tfidf = vec_desc.transform(test_df["lemmaized_wo_stopwords_raw_description"])

In [ ]:
train_dense = np.hstack([train_cat,train_bool])
test_dense = np.hstack([test_cat,test_bool])

X_train = hstack([
    csr_matrix(train_dense),
    train_name_tfidf,
    train_desc_tfidf
]).tocsr()
X_test = hstack([
    csr_matrix(test_dense),
    test_name_tfidf,
    test_desc_tfidf
]).tocsr()

y_train = train_df[target_col].values

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
lgb_params = {
    "objective": "mape",
    "metric": "mape",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbose": -1,
    "random_state": 42,
    "n_jobs": -1
}

In [ ]:
for fold, (train_idx, val_idx) in enumerate(
    kf.split(X_train, y_train)
):

    X_tr = X_train[train_idx]
    X_val = X_train[val_idx]

    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(stopping_rounds=100,verbose=False)]
    )

    oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds += (
        model.predict(
            X_test,
            num_iteration=model.best_iteration)/ kf.n_splits
    )

    fold_mape = np.mean(np.abs((y_val - oof_preds[val_idx]) / y_val)) * 100
    print(f"Fold {fold + 1}: {fold_mape:.4f}%")

In [ ]:
base_mape = np.mean(np.abs((y_train - oof_preds) / y_train)) * 100
print(f"\nBase OOF MAPE: {base_mape:.4f}%")

### улучшенная версия 21.6740%

In [ ]:
for df in [train_df, test_df]:
    df["exp_schedule"] = (
        df["experience_name"]
        + "_"
        + df["schedule_name"]
    )
    df["role_city"] = (
        df["professional_roles_name"]
        + "_"
        + df["unified_address_city"]
    )
    df["desc_len"] = (
        df["lemmaized_wo_stopwords_raw_description"]
        .apply(len)
    )
    df["skills_count"] = (
        df["key_skills_name"]
        .apply(
            lambda x:
            len(x.split(","))
            if x != "не указано"
            else 0
        )
    )

In [ ]:
extended_cat_cols = cat_cols + [
    "exp_schedule",
    "role_city"
]
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
train_cat = encoder.fit_transform(train_df[extended_cat_cols])
test_cat = encoder.transform(test_df[extended_cat_cols])

In [ ]:
num_cols = [
    "desc_len",
    "skills_count"
]

scaler = StandardScaler()
train_num = scaler.fit_transform(train_df[num_cols])
test_num = scaler.transform(test_df[num_cols])

train_dense = np.hstack([
    train_cat,
    train_bool,
    train_num
])
test_dense = np.hstack([
    test_cat,
    test_bool,
    test_num
])

In [ ]:
vec_name = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 3)
)
train_name_tfidf = vec_name.fit_transform(train_df["name_clean"])
test_name_tfidf = vec_name.transform(test_df["name_clean"])

In [ ]:
vec_desc = TfidfVectorizer(
    max_features=12000,
    min_df=5
)
train_desc_tfidf = vec_desc.fit_transform(
    train_df["lemmaized_wo_stopwords_raw_description"]
)
test_desc_tfidf = vec_desc.transform(
    test_df["lemmaized_wo_stopwords_raw_description"]
)

In [ ]:
X_train = hstack([
    csr_matrix(train_dense),
    train_name_tfidf,
    train_desc_tfidf
]).tocsr()

X_test = hstack([
    csr_matrix(test_dense),
    test_name_tfidf,
    test_desc_tfidf
]).tocsr()

In [ ]:
y_train_raw = train_df[target_col].values
y_train_log = np.log1p(y_train_raw)

In [ ]:
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.04,
    "num_leaves": 63,
    "max_depth": -1,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbose": -1,
    "random_state": 42,
    "n_jobs": -1
}

In [ ]:
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X_train, y_train_log)
):
    X_tr = X_train[train_idx]
    X_val = X_train[val_idx]

    y_tr = y_train_log[train_idx]
    y_val = y_train_log[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=4000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                verbose=False
            )
        ]
    )
    val_preds_log = model.predict(
        X_val,
        num_iteration=model.best_iteration
    )
    val_preds = np.expm1(val_preds_log)
    oof_preds[val_idx] = val_preds
    test_preds_log = model.predict(
        X_test,
        num_iteration=model.best_iteration
    )
    test_preds += (np.expm1(test_preds_log) / kf.n_splits)
    fold_mape = np.mean(
        np.abs((y_train_raw[val_idx] - val_preds) / y_train_raw[val_idx])) * 100

    print(f"Fold {fold + 1}: {fold_mape:.4f}%")

In [ ]:
advanced_mape = np.mean(
    np.abs((y_train_raw - oof_preds)/ y_train_raw)) * 100
print(f"\nAdvanced OOF MAPE: {advanced_mape:.4f}%")

In [ ]:
print(f"Base MAPE:      {base_mape:.4f}%")
print(f"Advanced MAPE:  {advanced_mape:.4f}%")
improvement = base_mape - advanced_mape
print(f"Improvement:    {improvement:.4f}%")

In [ ]:
submission = pd.DataFrame()
submission["id"] = test_df[id_col].astype(int)
submission["salary_mean_net"] = test_preds
submission.to_csv(
    "submission_lgb_mape.csv",
    index=False
)
print(submission.head())